# Playback saved model

In [ ]:
################################################################################
#                           1 Import packages                                  #
################################################################################

from amboworld.environment import Env

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random

import torch
import torch.nn as nn
import torch.optim as optim

# Use a double ended queue (deque) for memory
# When memory is full, this will replace the oldest value with the new one
from collections import deque

# Supress all warnings (e.g. deprecation warnings) for regular use
import warnings

warnings.filterwarnings("ignore")

In [ ]:
MODEL_PATH = './models/'

MODELS = {
    'ddqn':'ddqn_policy.pt',
    '3dqn': '3dqn_policy.pt'
}

# Replicate runs
TEST_EPISODES = 30

# Set whether to display on screen (slows model)
DISPLAY_ON_SCREEN = False

# SIM PARAMETERS
RANDOM_SEED = 42
SIM_DURATION = 5000
NUMBER_AMBULANCES = 3
NUMBER_INCIDENT_POINTS = 1
INCIDENT_RADIUS = 2
NUMBER_DISPTACH_POINTS = 25
AMBOWORLD_SIZE = 50
INCIDENT_INTERVAL = 60
EPOCHS = 2
AMBO_SPEED = 60
AMBO_FREE_FROM_HOSPITAL = False

In [ ]:
all_results = dict()

## Random action model

In [ ]:
results = dict()
results['call_to_arrival'] = []
results['assign_to_arrival'] = []
results['demand_met'] = []

sim = Env(
        random_seed = RANDOM_SEED,
        duration_incidents = SIM_DURATION,
        number_ambulances = NUMBER_AMBULANCES,
        number_incident_points = NUMBER_INCIDENT_POINTS,
        incident_interval = INCIDENT_INTERVAL,
        number_epochs = EPOCHS,
        number_dispatch_points = NUMBER_DISPTACH_POINTS,
        incident_range = INCIDENT_RADIUS,
        max_size = AMBOWORLD_SIZE,
        ambo_kph = AMBO_SPEED,
        ambo_free_from_hospital = AMBO_FREE_FROM_HOSPITAL
    )

for run in range(TEST_EPISODES):
    
    # Reset game environment and get first state observations
    state = sim.reset()

    # Continue loop until episode complete
    while True:    
        action = random.randint(0, sim.action_number -1)
        state_next, reward, terminal, info = sim.step(action)
        
        # Actions to take if end of game episode
        if terminal:
          
            print(f'Run: {run}, ', end='')
            mean_assignment_to_arrival = np.mean(info['assignment_to_arrival'])
            print(f'Mean assignment to arrival: {mean_assignment_to_arrival:4.1f}, ', end='')
            mean_call_to_arrival = np.mean(info['call_to_arrival'])
            print(f'Mean call to arrival: {mean_call_to_arrival:4.1f}, ', end='')
            demand_met = info['fraction_demand_met']
            print(f'Demand met {demand_met:0.3f}')
            
            results['call_to_arrival'].append(mean_call_to_arrival)
            results['assign_to_arrival'].append(mean_assignment_to_arrival)
            results['demand_met'].append(demand_met)
            
            break
            
results = pd.DataFrame(results)
filename = './output/results_random_action.csv'
results.to_csv(filename, index=False)

In [ ]:
results

In [ ]:
results.describe()